In [ ]:
# ==============================
# AgriIntel AI System
# Production-Ready ML Pipeline
# ==============================

import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

print("Environment Ready ✅")

Environment Ready ✅


In [ ]:
# ==============================
# Step 1: Load Dataset
# ==============================
from google.colab import files
import pandas as pd
import os

# Upload file using button
uploaded = files.upload()

# Get uploaded file name
file_name = list(uploaded.keys())[0]

# Load dataset
df = pd.read_csv(file_name)

# Display basic info
print("Dataset loaded successfully!")
print("File name:", file_name)
print("Shape:", df.shape)


# ==============================
# Step 2: Basic Data Validation
# ==============================

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

# Ensure correct column order
expected_columns = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']

assert list(df.columns) == expected_columns, "Column structure mismatch!"

print("\nColumn Structure Verified ✅")

Saving Crop_recommendation.csv to Crop_recommendation.csv
Dataset loaded successfully!
File name: Crop_recommendation.csv
Shape: (2200, 8)

Missing Values:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Data Types:
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label           object
dtype: object

Column Structure Verified ✅


In [ ]:
# ==============================
# Advanced Crop Model Training
# ==============================

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform

# Features & Target
X_crop = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y_crop = df['label']

# Encode labels
crop_encoder = LabelEncoder()
y_crop_encoded = crop_encoder.fit_transform(y_crop)

# Define XGBoost base model
xgb_base = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42
)

# Hyperparameter space
param_dist = {
    'n_estimators': randint(200, 500),
    'max_depth': randint(4, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# Fit Search
random_search.fit(X_crop, y_crop_encoded)

best_xgb = random_search.best_estimator_

# Calibrate Probabilities
calibrated_crop_model = CalibratedClassifierCV(
    best_xgb,
    method='sigmoid',
    cv=5
)

calibrated_crop_model.fit(X_crop, y_crop_encoded)

print("Advanced Crop Model Training Completed ✅")
print("Best Parameters:", random_search.best_params_)

# Cross-validation accuracy
cv_scores = random_search.cv_results_['mean_test_score']
print("Best CV Accuracy:", max(cv_scores))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Advanced Crop Model Training Completed ✅
Best Parameters: {'colsample_bytree': np.float64(0.6624074561769746), 'learning_rate': np.float64(0.041198904067240534), 'max_depth': 6, 'n_estimators': 287, 'subsample': np.float64(0.7334834444556088)}
Best CV Accuracy: 0.9954545454545454


In [ ]:
# ==============================
# Production Inference Utilities (Fixed)
# ==============================

def predict_top3_crops(model, encoder, input_array):
    """
    Returns top 3 crop predictions with probabilities.
    """
    probs = model.predict_proba(input_array)[0]
    top3_idx = np.argsort(probs)[-3:][::-1]

    crops = encoder.inverse_transform(top3_idx)
    confidences = probs[top3_idx]

    return list(zip(crops, confidences))


def get_crop_feature_importance(model, feature_names):
    """
    Extract feature importance from calibrated model.
    Handles CalibratedClassifierCV properly.
    """

    # CalibratedClassifierCV stores trained estimators here
    calibrated_models = model.calibrated_classifiers_

    # Take first fold's estimator (they are similar after calibration)
    xgb_model = calibrated_models[0].estimator

    importances = xgb_model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    return importance_df


print("Inference Utilities Updated ✅")

Inference Utilities Updated ✅


In [ ]:
# ==============================
# Test Top-3 Prediction Pipeline
# ==============================

# Take a random sample
sample = X_crop.sample(1, random_state=42)
sample_array = sample.values

top3_predictions = predict_top3_crops(
    calibrated_crop_model,
    crop_encoder,
    sample_array
)

print("Top 3 Predictions for Sample:")
for crop, conf in top3_predictions:
    print(f"{crop} → {conf:.4f}")

Top 3 Predictions for Sample:
muskmelon → 0.9464
rice → 0.0035
lentil → 0.0031


In [ ]:
feature_names = X_crop.columns.tolist()

importance_df = get_crop_feature_importance(
    calibrated_crop_model,
    feature_names
)

display(importance_df)

,Feature,Importance
2,K,0.186526
4,humidity,0.172362
6,rainfall,0.170921
1,P,0.150746
0,N,0.139912
3,temperature,0.105429
5,ph,0.074103


In [ ]:
# ==============================
# Step 4: Compute Crop-wise Nutrient Baselines
# ==============================

crop_nutrient_means = df.groupby("label")[["N", "P", "K"]].mean()

print("Crop-wise Nutrient Baselines Computed ✅")
display(crop_nutrient_means.head())

Crop-wise Nutrient Baselines Computed ✅


,N,P,K
label,,,
apple,20.80,134.22,199.89
banana,100.23,82.01,50.05
blackgram,40.02,67.47,19.24
chickpea,40.09,67.79,79.92
coconut,21.98,16.93,30.59


In [ ]:
# ==============================
# Step 5: Engineer Fertilizer Labels
# ==============================

def assign_fertilizer_class(row):
    crop = row["label"]
    mean_values = crop_nutrient_means.loc[crop]

    delta_N = mean_values["N"] - row["N"]
    delta_P = mean_values["P"] - row["P"]
    delta_K = mean_values["K"] - row["K"]

    deltas = {
        "Nitrogen_Rich": delta_N,
        "Phosphorus_Rich": delta_P,
        "Potassium_Rich": delta_K
    }

    # Get most deficient nutrient
    max_deficiency = max(deltas, key=deltas.get)

    # If no deficiency (all deltas negative or small)
    if all(v <= 0 for v in deltas.values()):
        return "Balanced_NPK"

    return max_deficiency


df["fertilizer_class"] = df.apply(assign_fertilizer_class, axis=1)

print("Fertilizer Classes Engineered Successfully ✅")
print("\nClass Distribution:")
print(df["fertilizer_class"].value_counts())

Fertilizer Classes Engineered Successfully ✅

Class Distribution:
fertilizer_class
Nitrogen_Rich      870
Phosphorus_Rich    677
Potassium_Rich     368
Balanced_NPK       285
Name: count, dtype: int64


In [ ]:
# ==============================
# Step 6: Prepare Fertilizer Model Data
# ==============================

# Encode crop label for fertilizer model input
fert_crop_encoder = LabelEncoder()
df["crop_encoded"] = fert_crop_encoder.fit_transform(df["label"])

# Encode fertilizer class
fert_encoder = LabelEncoder()
df["fertilizer_encoded"] = fert_encoder.fit_transform(df["fertilizer_class"])

# Features for fertilizer prediction
X_fert = df[[
    "N", "P", "K",
    "temperature", "humidity", "rainfall",
    "crop_encoded"
]]

y_fert = df["fertilizer_encoded"]

print("Fertilizer Model Data Prepared ✅")
print("Feature Shape:", X_fert.shape)
print("Target Classes:", fert_encoder.classes_)

Fertilizer Model Data Prepared ✅
Feature Shape: (2200, 7)
Target Classes: ['Balanced_NPK' 'Nitrogen_Rich' 'Phosphorus_Rich' 'Potassium_Rich']


In [ ]:
# ==============================
# Step 7: Advanced Fertilizer Model Training
# ==============================

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

rf_base = RandomForestClassifier(random_state=42)

param_dist_rf = {
    "n_estimators": randint(200, 600),
    "max_depth": randint(5, 20),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5)
}

skf_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=20,
    cv=skf_rf,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search_rf.fit(X_fert, y_fert)

best_rf = random_search_rf.best_estimator_

print("Advanced Fertilizer Model Training Completed ✅")
print("Best Parameters:", random_search_rf.best_params_)
print("Best CV Accuracy:", random_search_rf.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Advanced Fertilizer Model Training Completed ✅
Best Parameters: {'max_depth': 16, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 513}
Best CV Accuracy: 0.9168181818181818


In [ ]:
# ==============================
# Fertilizer Inference Utilities
# ==============================

def predict_fertilizer(model, encoder, input_array):
    probs = model.predict_proba(input_array)[0]
    top_idx = np.argmax(probs)

    fert_label = encoder.inverse_transform([top_idx])[0]
    confidence = probs[top_idx]

    return fert_label, confidence


def get_fertilizer_feature_importance(model, feature_names):
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    return importance_df


# Test with sample row
sample_fert = X_fert.sample(1, random_state=42)
sample_array = sample_fert.values

fert_label, fert_conf = predict_fertilizer(
    best_rf,
    fert_encoder,
    sample_array
)

print("Sample Fertilizer Prediction:")
print(f"{fert_label} → {fert_conf:.4f}")

fert_importance_df = get_fertilizer_feature_importance(
    best_rf,
    X_fert.columns.tolist()
)

display(fert_importance_df)

Sample Fertilizer Prediction:
Potassium_Rich → 0.8760


,Feature,Importance
0,N,0.324902
1,P,0.221006
2,K,0.140302
4,humidity,0.094702
5,rainfall,0.089877
3,temperature,0.077314
6,crop_encoded,0.051897


In [ ]:
# ==============================
# Step 8: Save Models for Deployment
# ==============================

import os

# Create models directory if not exists
os.makedirs("models", exist_ok=True)

# Save Crop Model
pickle.dump(calibrated_crop_model, open("models/crop_model.pkl", "wb"))
pickle.dump(crop_encoder, open("models/crop_encoder.pkl", "wb"))

# Save Fertilizer Model
pickle.dump(best_rf, open("models/fertilizer_model.pkl", "wb"))
pickle.dump(fert_encoder, open("models/fertilizer_encoder.pkl", "wb"))
pickle.dump(fert_crop_encoder, open("models/fert_crop_encoder.pkl", "wb"))

print("All Models Saved Successfully in /models folder ✅")

All Models Saved Successfully in /models folder ✅


In [ ]:
# ==============================
# Save Feature Metadata
# ==============================

metadata = {
    "crop_features": X_crop.columns.tolist(),
    "fertilizer_features": X_fert.columns.tolist()
}

pickle.dump(metadata, open("models/metadata.pkl", "wb"))

print("Metadata Saved Successfully ✅")

Metadata Saved Successfully ✅


In [ ]:
import os
os.listdir("models")

['crop_model.pkl',
 'fertilizer_model.pkl',
 'metadata.pkl',
 'fert_crop_encoder.pkl',
 'crop_encoder.pkl',
 'fertilizer_encoder.pkl']

In [ ]:
from google.colab import files
import shutil

# Zip models folder
shutil.make_archive("models_backup", 'zip', "models")

# Download zip
files.download("models_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib
import os

os.makedirs("models_small", exist_ok=True)

joblib.dump(calibrated_crop_model, "models_small/crop_model.pkl", compress=3)
joblib.dump(crop_encoder, "models_small/crop_encoder.pkl", compress=3)

joblib.dump(best_rf, "models_small/fertilizer_model.pkl", compress=3)
joblib.dump(fert_encoder, "models_small/fertilizer_encoder.pkl", compress=3)
joblib.dump(fert_crop_encoder, "models_small/fert_crop_encoder.pkl", compress=3)
joblib.dump(metadata, "models_small/metadata.pkl", compress=3)

print("Compressed models saved.")

Compressed models saved.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("models_small_backup", 'zip', "models_small")
files.download("models_small_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>